## Sensor Data Normalization (Min-Max Scaling)
**Scenario**: You have an array of raw sensor readings and need to scale all values to a range between 0 and 1.

**Task**: Write a function that takes a 1D array of numbers and scales them using the formula:

$$\text{scaled} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$


In [16]:

import numpy as np
from numpy.typing import NDArray


def min_max_normalize(data: NDArray[np.float32]) -> NDArray[np.float32]:
    min_v = np.min(data)
    max_v = np.max(data)

    return (data - min_v) / delta if (delta := max_v - min_v) != 0 else np.zeros_like(data)




In [17]:
min_max_normalize(np.array([1., 2., 3., 4.]))

array([0.        , 0.33333333, 0.66666667, 1.        ])

---


## Moving Average Smoothing
**Scenario**: A time-series of temperature measurements contains high-frequency noise. You need to smooth the curve using a rolling window of size 3.

**Task**: Use np.convolve to compute a moving average. Ensure the output length matches the input or handles boundaries cleanly (using the 'valid' mode).

In [18]:
def moving_average(data: NDArray[np.float32], sliding_window: int = 3) -> NDArray[np.float32]:
    sec = np.ones(sliding_window) / sliding_window
    return np.convolve(data, sec, mode='valid')


temperatures = np.array([20.1, 20.5, 21.0, 22.2, 21.8, 22.5, 23.0])
moving_average(temperatures)



array([20.53333333, 21.23333333, 21.66666667, 22.16666667, 22.43333333])

---

## Outlier Filtering via Z-Scores
**Scenario**: You need to clean a dataset by removing values that deviate significantly from the mean.

**Task**: Compute the Z-score for each element in an array (how many standard deviations away from the mean it is) and return a boolean mask for elements where the absolute Z-score is less than or equal to 2.



In [19]:
def filter_outliers(data: NDArray[np.float32], threshold=2.0) -> NDArray[np.bool_]:
    mean = np.mean(data)
    std = np.std(data)

    if std == 0:
        return np.zeros_like(data, dtype=np.bool_)
    z_scores = (data - mean) / std
    return np.abs(z_scores) <= threshold


measurements = np.array([10.2, 9.8, 10.1, 25.5, 10.3, 9.9, -5.0])
mask = filter_outliers(measurements, threshold=2.0)

measurements[mask]



array([10.2,  9.8, 10.1, 25.5, 10.3,  9.9, -5. ])

---

## Multi-Sensor Industrial Log Analysis
**Scenario**: You have a 2D NumPy array representing sensor logs from a production line over 100 time steps. The array shape is (100, 4), where the columns correspond to:
- Temperature (°C)
- Vibration ($\text{mm/s}$)
- Pressure ($\text{bar}$)
- Current ($\text{A}$)

**Tasks**
**Column-wise Statistics:** Compute the mean and maximum values for each of the four sensors independently using the axis parameter.

**Peak Vibration Index:** Find the row index (time step) that recorded the absolute highest vibration value.

**Safety Threshold Mask:** Create a boolean mask identifying all time steps where any sensor exceeds its critical threshold:
 - Temperature > 82.0
 - Vibration > 6.0
 - Pressure > 48.0
 - Current > 15.0

In [22]:
import numpy as np

np.random.seed(42)
n_steps = 100
temp = np.random.normal(70,5, n_steps)
vibration = np.random.exponential(1.5,n_steps)
pressure = np.random.uniform(30.0, 50.0, n_steps)
current = np.random.normal(12.0, 1.2, n_steps)

sensor_data = np.round(np.column_stack((temp, vibration, pressure, current)),decimals=1)
means = np.round(np.mean(sensor_data, axis=0),decimals=1)
maxs = np.round(np.max(sensor_data, axis=0),decimals=1)

print(f"means: {means}, maxs: {maxs}")

peak_vibr_idx = np.argmax(sensor_data[:, 1], axis=0,)
peak_vibr_val = sensor_data[peak_vibr_idx, 1]

print(f" vibr idx = {peak_vibr_idx} and val {peak_vibr_val}")

thresholds = np.array([82.0, 6.0, 48.0, 15.0])
critical_events = sensor_data[np.any(sensor_data > thresholds, axis=1)]

print(f"critical events: {critical_events}")


means: [69.5  1.4 40.6 12. ], maxs: [79.3  6.4 49.8 14.6]
 vibr idx = 24 and val 6.4
critical events: [[67.7  4.9 48.8 14. ]
 [67.7  0.4 49.1 10.3]
 [71.2  1.  48.3 11.5]
 [67.2  0.1 48.6 10.3]
 [71.6  1.  49.3 12. ]
 [65.5  0.1 49.3 10.8]
 [67.3  6.4 33.4 12.1]
 [64.2  1.7 48.7 13.9]
 [79.3  1.5 49.8 12.7]
 [68.5  0.4 48.3 12.8]
 [74.1  0.  49.5 10.9]
 [67.4  0.4 49.3 11.3]
 [68.   0.5 49.1 12.4]]
